### Một cơ hội mua có thể xuất hiện khi giá cổ phiếu chạm vào hoặc tiệm cận Lower Bollinger Band, đồng thời RSI ở dưới 30.
#### Một cơ hội bán tốt có thể xuất hiện khi giá chạm vào hoặc tiệm cận Upper Bollinger Band đồng thời RSI ở trên 70.

#### Tao cac cot BB...
#### Tạo cột tín hiệu mua và bán
#### data['Buy_Signal'] = ((data['Close'] <= data['Lower_Band']) & (data['RSI'] < 30))
#### data['Sell_Signal'] = ((data['Close'] >= data['Upper_Band']) & (data['RSI'] > 70))

!["BB RSI"](./FlowChart/flowchart_bb_rsi_signal.png)

# Ham o muc FX.1

In [1]:
def loaddataMT5_FromTo(symbol, from_date, to_date, timeframe):
    import MetaTrader5 as mt5
    from datetime import datetime
    import pandas as pd 

    # Kết nối tới MetaTrader 5
    if not mt5.initialize():
        print("Khởi tạo MT5 không thành công")
        mt5.shutdown()

    from_date_str = datetime.strptime(from_date, '%Y-%m-%d')
    to_date_str = datetime.strptime(to_date, '%Y-%m-%d')
    
    # Lấy dữ liệu OHLC cho cặp tiền symbol trong khoảng thời gian đã xác định
    ohlc_data = mt5.copy_rates_range(symbol, timeframe, from_date_str, to_date_str)
    # Ngắt kết nối với MT5
    mt5.shutdown()

    # Chuyển dữ liệu nhận được thành DataFrame và hiển thị
    data = pd.DataFrame(ohlc_data)
    data['time'] = pd.to_datetime(data['time'], unit='s')

    # ohlc_df.reset_index(inplace=True)

    data = data.rename(columns={'time': 'Datetime'})        
    data = data.rename(columns={'open': 'Open'})       
    data = data.rename(columns={'high': 'High'})       
    data = data.rename(columns={'low': 'Low'})       
    data = data.rename(columns={'close': 'Close'})       
    data = data.rename(columns={'tick_volume': 'Volume'})       

    data = pd.DataFrame(data, columns=['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume'])

    return data

# Dua tat ca cac thong tin o tren de lam chien luoc

In [5]:
import MetaTrader5 as mt5
import pandas as pd
import plotly.graph_objects as go
import redis
import numpy as np
from datetime import datetime, timedelta
import talib

# ##############################################Step 0: Các tham số để lấy dữ liệu###############################
symbol = 'EURUSD.sml'
from_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=0)).strftime('%Y-%m-%d')
timeframe = mt5.TIMEFRAME_D1

# ##############################################Step 1: Lấy dữ liệu##############################################
data = loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)
# Thiết lập 'Datetime' làm chỉ mục của DataFrame
data.set_index('Datetime', inplace=True)

# ##############################################Step 2: Chiến lược##############################################  

# Tính toán Bollinger Bands bằng TA-Lib
window = 20
data['Upper_Band'], data['Middle_Band'], data['Lower_Band'] = talib.BBANDS(data['Close'], timeperiod=window, nbdevup=2, nbdevdn=2, matype=0)

# Tính RSI bằng TA-Lib
windowRSI = 17
data['RSI'] = talib.RSI(data['Close'], timeperiod=windowRSI)

# MFI
# data['MFI'] = talib.MFI(data['High'], data['Low'], data['Close'], data['Volume'], timeperiod=14)

# Tạo cột tín hiệu mua và bán
data['Buy_Signal'] = ((data['Close'] <= data['Lower_Band']) & (data['RSI'] < 30)) # & ((np.maximum(data['Open'], data['Close'])) >= data['Lower_Band']))
data['Sell_Signal'] = ((data['Close'] >= data['Upper_Band']) & (data['RSI'] > 70))

In [6]:
data

,Open,High,Low,Close,Volume,Upper_Band,Middle_Band,Lower_Band,RSI,Buy_Signal,Sell_Signal
Datetime,,,,,,,,,,,
2025-07-01,1.17848,1.18300,1.17614,1.18058,161221,NaN,NaN,NaN,NaN,False,False
2025-07-02,1.18024,1.18101,1.17470,1.17987,152126,NaN,NaN,NaN,NaN,False,False
2025-07-03,1.17968,1.18102,1.17175,1.17576,133819,NaN,NaN,NaN,NaN,False,False
2025-07-04,1.17591,1.17877,1.17535,1.17772,127897,NaN,NaN,NaN,NaN,False,False
2025-07-07,1.17858,1.17903,1.16867,1.17086,136268,NaN,NaN,NaN,NaN,False,False
2025-07-08,1.17092,1.17654,1.16825,1.17258,149208,NaN,NaN,NaN,NaN,False,False
2025-07-09,1.17210,1.17294,1.16897,1.17228,131627,NaN,NaN,NaN,NaN,False,False
2025-07-10,1.17188,1.17498,1.16627,1.17006,117351,NaN,NaN,NaN,NaN,False,False
2025-07-11,1.16992,1.17142,1.16648,1.16910,144646,NaN,NaN,NaN,NaN,False,False


In [18]:
pip show talib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
data

In [ ]:
data.to_csv('Buoi4.3_FX_Chienluoc.csv')

In [ ]:
import pandas as pd
import plotly.graph_objects as go

####################################################################################################
# Tạo figure
fig = go.Figure()

# Thêm đường giá đóng cửa, SMA và Bollinger Bands
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Giá Đóng Cửa'))
# fig.add_trace(go.Scatter(x=data.index, y=data['SMA'], mode='lines', name='SMA'))
fig.add_trace(go.Scatter(x=data.index, y=data['Upper_Band'], mode='lines', name='Bollinger Band Trên', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=data.index, y=data['Lower_Band'], mode='lines', name='Bollinger Band Dưới', line=dict(dash='dash')))

# Điền màu giữa Upper và Lower Bands
fig.add_trace(go.Scatter(x=data.index, y=data['Upper_Band'], mode='lines', name='Upper Band', line=dict(width=0)))
fig.add_trace(go.Scatter(x=data.index, y=data['Lower_Band'], mode='lines', name='Lower Band', fill='tonexty', fillcolor='rgba(204,204,204,0.2)', line=dict(width=0)))

# Vẽ các điểm cho tín hiệu mua
buy_signals = data[data['Buy_Signal']]
fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(symbol='triangle-up', size=10, color='green')))

# Vẽ các điểm cho tín hiệu bán
sell_signals = data[data['Sell_Signal']]
fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(symbol='triangle-down', size=10, color='red')))

# Cập nhật thông tin layout
fig.update_layout(title='Biểu Đồ Bollinger Bands và RSI', xaxis_title='Ngày', yaxis_title='Giá', showlegend=True)

# Vẽ biểu đồ
fig.show()

####################################################################################################
# Tạo figure
fig = go.Figure()

fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], mode='lines', name='RSI', line=dict(color='purple')))

# Thêm các đường ngang tại các mức 70 và 30
fig.add_hline(y=70, line_dash="dash", line_color="red", line_width=0.5)
fig.add_hline(y=30, line_dash="dash", line_color="green", line_width=0.5)

# Điền màu giữa các mức 70 và 30
fig.add_trace(go.Scatter(x=data.index, y=[70]*len(data.index), mode='lines', name='RSI70', line=dict(width=0)))
fig.add_trace(go.Scatter(x=data.index, y=[30]*len(data.index), mode='lines', name='RSI30', fill='tonexty', fillcolor='rgba(204,204,204,0.2)', line=dict(width=0)))

# Cập nhật thông tin layout cho biểu đồ RSI
fig.update_layout(title='Biểu Đồ RSI', xaxis_title='Ngày', yaxis_title='RSI', showlegend=True)

# Vẽ biểu đồ
fig.show()

: 

In [ ]:
import numpy as np
print(np.__version__)
